In [ ]:
!pip install yfinance==0.2.66

import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import time
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

ticker = "JPM"

end_date = datetime.now()
start_date = end_date - timedelta(days=7)   # Yahoo only serves 1m data for ~trailing 7 days

all_data = pd.DataFrame()
current_end = end_date
max_retries = 5
base_delay = 10
chunk_delay = 5
chunk_days = 1

def is_weekend_only(start, end):
    d = start
    while d < end:
        if d.weekday() < 5:  # Mon-Fri
            return False
        d += timedelta(days=1)
    return True

while current_end > start_date:
    current_start = max(current_end - timedelta(days=chunk_days), start_date)

    if is_weekend_only(current_start, current_end):
        logger.info(f"Skipping {current_start} to {current_end} — weekend, market closed.")
        current_end = current_start
        continue

    logger.info(f"Fetching data from {current_start} to {current_end}")

    for attempt in range(max_retries):
        try:
            chunk = yf.download(ticker, start=current_start, end=current_end, interval="1m", auto_adjust=False)

            if not chunk.empty:
                all_data = pd.concat([all_data, chunk])
                logger.info(f"Successfully fetched data for {current_start} to {current_end}")
                break
            else:
                logger.warning(f"Empty data for {current_start} to {current_end}, retrying...")

        except Exception as e:
            logger.error(f"Error on attempt {attempt + 1}/{max_retries}: {e}")

        if attempt < max_retries - 1:
            delay = base_delay * (2 ** attempt)
            logger.info(f"Waiting {delay} seconds before retrying...")
            time.sleep(delay)
        else:
            logger.error(f"Max retries reached for {current_start} to {current_end}, skipping...")

        time.sleep(2)

    time.sleep(chunk_delay)
    current_end = current_start

all_data = all_data[~all_data.index.duplicated(keep='first')]
all_data = all_data.sort_index()

print(f"Total rows fetched: {len(all_data)}")
print(all_data.head())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Total rows fetched: 1950
Price                       Adj Close       Close        High         Low  \
Ticker                            JPM         JPM         JPM         JPM   
Datetime                                                                    
2026-09-14 13:30:00+00:00  354.399994  354.399994  355.229889  354.169708   
2026-09-14 13:31:00+00:00  354.730011  354.730011  355.279999  354.220001   
2026-09-14 13:32:00+00:00  354.480011  354.480011  354.890015  353.630005   
2026-09-14 13:33:00+00:00  354.040100  354.040100  354.744995  353.769989   
2026-09-14 13:34:00+00:00  353.640015  353.640015  354.820007  353.640015   

Price                            Open  Volume  
Ticker                            JPM     JPM  
Datetime                                       
2026-09-14 13:30:00+00:00  354.230011  142532  
2026-09-14 13:31:00+00:00  354.220001   23484  
2026-09-14 13:32:00+00:00  354.619995   24981  
2026-09-14 13:33:00+00:00  354.454987    9431  
2026-09-14 13:34:00+00

In [ ]:
# Flatten the MultiIndex columns (Price/Ticker) down to simple names
# before saving, so the CSV loads cleanly with a plain pd.read_csv() later.
if isinstance(all_data.columns, pd.MultiIndex):
    all_data.columns = all_data.columns.get_level_values(0)

all_data = all_data.reset_index()  # pulls Datetime out of the index into a real column
all_data = all_data.rename(columns={'Datetime': 'Timestamp'})
all_data.columns.name = None

# Reorder so Timestamp is first
cols = ['Timestamp'] + [c for c in all_data.columns if c != 'Timestamp']
all_data = all_data[cols]

filename = f"{ticker}.csv"
all_data.to_csv(filename, index=False)
logger.info(f"Data saved to {filename} (flattened, {len(all_data)} rows)")

print(all_data.head())
print(all_data.dtypes)

                  Timestamp   Adj Close       Close        High         Low  \
0 2026-09-14 13:30:00+00:00  354.399994  354.399994  355.229889  354.169708   
1 2026-09-14 13:31:00+00:00  354.730011  354.730011  355.279999  354.220001   
2 2026-09-14 13:32:00+00:00  354.480011  354.480011  354.890015  353.630005   
3 2026-09-14 13:33:00+00:00  354.040100  354.040100  354.744995  353.769989   
4 2026-09-14 13:34:00+00:00  353.640015  353.640015  354.820007  353.640015   

         Open  Volume  
0  354.230011  142532  
1  354.220001   23484  
2  354.619995   24981  
3  354.454987    9431  
4  354.489990   20463  
Timestamp    datetime64[ns, UTC]
Adj Close                float64
Close                    float64
High                     float64
Low                      float64
Open                     float64
Volume                     int64
dtype: object


In [ ]:
from google.colab import files
files.download(filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>